# UnifyWeaver वंशावली वृक्ष ट्यूटोरियल

यह इंटरैक्टिव नोटबुक प्रदर्शित करती है कि Prolog प्रेडिकेट्स को Bash स्क्रिप्ट में संकलित करने के लिए UnifyWeaver का उपयोग कैसे करें।

## पूर्वापेक्षाएँ

- SWI-Prolog स्थापित हो
- UnifyWeaver लाइब्रेरी उपलब्ध हो
- Prolog Jupyter कर्नेल स्थापित हो (`pip install prolog-jupyter-kernel`)

## सीखने के उद्देश्य

इस नोटबुक के अंत तक, आप निम्न में सक्षम होंगे:
1. Prolog तथ्य और नियम परिभाषित करना
2. प्रेडिकेट्स को Bash में संकलित करने के लिए UnifyWeaver का उपयोग करना
3. जनरेट की गई Bash स्क्रिप्ट का परीक्षण करना
4. ट्रांजिटिव क्लोजर संकलन को समझना

## चरण 1: UnifyWeaver वातावरण प्रारंभ करें

सबसे पहले, हमें UnifyWeaver मॉड्यूल लोड करने होंगे। हम education निर्देशिका से `init.pl` फ़ाइल का उपयोग करेंगे।

In [ ]:
% आरंभीकरण फ़ाइल लोड करें
['../init'].

## चरण 2: पारिवारिक संबंध परिभाषित करें

आइए बाइबिल वंशावली वृक्ष से कुछ माता-पिता और बच्चों के संबंध परिभाषित करें।

In [ ]:
% parent तथ्य परिभाषित करें
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## चरण 3: अभिभावक प्रश्नों का परीक्षण करें

संकलन से पहले, आइए कुछ Prolog प्रश्नों के साथ सत्यापित करें कि हमारा डेटा सही है।

In [ ]:
% प्रश्न: अब्राहम के बच्चे कौन हैं?
parent(abraham, Child).

In [ ]:
% प्रश्न: याकूब के बच्चे कौन हैं?
parent(jacob, Child).

## चरण 4: पूर्वज संबंध परिभाषित करें

अब हम ट्रांजिटिव क्लोजर — `ancestor` संबंध — को परिभाषित करते हैं।

In [ ]:
% ancestor को parent के ट्रांजिटिव क्लोजर के रूप में परिभाषित करें
:- dynamic ancestor/2.

% आधार स्थिति: माता-पिता एक पूर्वज हैं
ancestor(X, Y) :- parent(X, Y).

% पुनरावर्ती स्थिति: यदि X, Y का माता-पिता है और Y, Z का पूर्वज है, तो X, Z का पूर्वज है
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## चरण 5: पूर्वज प्रश्नों का परीक्षण करें

आइए सत्यापित करें कि हमारा पूर्वज प्रेडिकेट सही ढंग से काम करता है।

In [ ]:
% प्रश्न: क्या अब्राहम याकूब का पूर्वज है?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% प्रश्न: अब्राहम के सभी वंशज कौन हैं?
ancestor(abraham, Descendant).

## चरण 6: Parent को Bash में संकलित करें

अब मजेदार हिस्सा — आइए अपने `parent/2` तथ्यों को एक Bash स्क्रिप्ट में संकलित करें!

In [ ]:
% स्ट्रीम कंपाइलर लोड करें
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % parent तथ्यों को bash में संकलित करें
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## चरण 7: Parent स्क्रिप्ट सहेजें

आइए जनरेट किए गए Bash कोड को एक फ़ाइल में सहेजें।

In [ ]:
% फ़ाइल में सहेजें
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## चरण 8: Ancestor को Bash में संकलित करें

अब `ancestor/2` प्रेडिकेट को संकलित करें, जो रिकर्सन का उपयोग करता है।

In [ ]:
% पुनरावर्ती कंपाइलर लोड करें
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % ancestor को bash में संकलित करें
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## चरण 9: Ancestor स्क्रिप्ट सहेजें

पूर्वज स्क्रिप्ट को एक फ़ाइल में सहेजें।

In [ ]:
% फ़ाइल में सहेजें
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## चरण 10: जनरेट की गई स्क्रिप्ट का परीक्षण करें

अब अपनी जनरेट की गई Bash स्क्रिप्ट का परीक्षण करें! हम bash कमांड चलाने के लिए `%%bash` मैजिक का उपयोग करेंगे।

In [ ]:
%%bash
# parent स्क्रिप्ट को source करें
source ../output/parent.sh

# परीक्षण: अब्राहम के बच्चे कौन हैं?
echo "अब्राहम के बच्चे:"
parent abraham

In [ ]:
%%bash
# दोनों स्क्रिप्ट्स को source करें
source ../output/parent.sh
source ../output/ancestor.sh

# परीक्षण: अब्राहम के वंशज कौन हैं?
echo "अब्राहम के वंशज:"
ancestor abraham

In [ ]:
%%bash
# दोनों स्क्रिप्ट्स को source करें
source ../output/parent.sh
source ../output/ancestor.sh

# परीक्षण: क्या अब्राहम यहूदा का पूर्वज है?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ हाँ, अब्राहम यहूदा का पूर्वज है"
else
    echo "✗ नहीं"
fi

## चरण 11: संकलन रणनीति को समझना

आइए विश्लेषण करें कि UnifyWeaver ने क्या किया:

1. **Parent संकलन**: सभी माता-पिता-बच्चे जोड़े उत्सर्जित करने वाले एक साधारण स्ट्रीमिंग फ़ंक्शन को बनाने के लिए `stream_compiler` का उपयोग किया

2. **Ancestor संकलन**: ट्रांजिटिव क्लोजर पैटर्न का पता लगाया और सभी सुलभ पूर्वजों की कुशलतापूर्वक गणना करने के लिए BFS (चौड़ाई-प्रथम खोज) अनुकूलन लागू किया

आइए संकलन रणनीति की पुष्टि करें:

In [ ]:
% जाँचें कि क्या ancestor को पुनरावर्ती के रूप में वर्गीकृत किया गया है
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## सारांश

इस नोटबुक में, आपने सीखा:

✅ Prolog तथ्य और नियम कैसे परिभाषित करें

✅ तथ्यों के लिए UnifyWeaver के `stream_compiler` का उपयोग कैसे करें

✅ पुनरावर्ती प्रेडिकेट्स के लिए UnifyWeaver के `recursive_compiler` का उपयोग कैसे करें

✅ जनरेट की गई Bash स्क्रिप्ट का परीक्षण कैसे करें

✅ UnifyWeaver स्वचालित रूप से ट्रांजिटिव क्लोजर का पता लगाता है और BFS अनुकूलन लागू करता है

## अगले कदम

इन अभ्यासों को आजमाएं:

1. वंशावली वृक्ष में परिवार के और अधिक सदस्य जोड़ें
2. `grandparent/2` प्रेडिकेट परिभाषित करें और इसे संकलित करें
3. एक `sibling/2` प्रेडिकेट बनाएं (एक ही माता-पिता वाले दो लोग)
4. BFS एल्गोरिथ्म को समझने के लिए जनरेट किए गए Bash कोड का अन्वेषण करें

उन्नत रिकर्सन पैटर्न के बारे में जानने के लिए **नोटबुक 2: रिकर्सन पैटर्न तुलना** पर आगे बढ़ें!